In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import time

In [0]:
df = spark.read.format("parquet").load(f"abfss://bronze@projdatalake.dfs.core.windows.net/products")
df.display()

In [0]:
df.printSchema()

In [0]:
# Remove last column
df = df.drop('_rescued_data')

df.display()

#### **Functions**

(i) Using sql

In [0]:
# Lets create a temp view to apply our function
df.createOrReplaceTempView("products")

In [0]:
%sql
-- function to apply 10% discount to price
CREATE OR REPLACE FUNCTION databricks_wspace.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN p_price * 0.9

In [0]:
%sql
select product_id, price, databricks_wspace.bronze.discount_func(price) as discounted_price
from products

Applying the function to dataframe directly

In [0]:
# here we have to use expr.
df = df.withColumn('discounted_price', expr("databricks_wspace.bronze.discount_func(price)"))
df.display()

(ii) Using Python

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_wspace.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS
$$
    return p_brand.upper()
$$

In [0]:
%sql
select product_id, brand, databricks_wspace.bronze.upper_func(brand) as brand_upper
from products

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@projdatalake.dfs.core.windows.net/products")

Next create tables on top of our data

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_wspace.silver.products_silver
USING DELTA
LOCATION 'abfss://silver@projdatalake.dfs.core.windows.net/products'

In [0]:
%sql
select * from databricks_wspace.silver.products_silver